<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/CIFAR10_TensorBoard_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extract CIFAR-10 PEPITA metrics

This notebook reads the TensorBoard event file from the completed CIFAR-10 run and exports the logged scalar metrics for the report. It does not rerun training or change the experiment.

In Colab, upload the event file from `experiments/cifar10_exp01_pepita_1hidden/logs/tb_logs/version_0/`. Its filename begins with `events.out.tfevents`. The run logged the test set through the project's validation hook, so the reported `val_acc` is test-set accuracy monitored each epoch, not a separate validation-set score.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('tensorboard') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorboard'])
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
print('TensorBoard is ready. Python:', sys.version.split()[0])

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
event_files = [name for name in uploaded if 'tfevents' in name]
if len(event_files) != 1:
    raise ValueError(f'Upload exactly one TensorBoard event file; found {len(event_files)}.')
event_path = Path(event_files[0])
print('Reading:', event_path.name, 'Size:', round(event_path.stat().st_size / 1024, 1), 'KB')

In [ ]:
accumulator = EventAccumulator(str(event_path), size_guidance={'scalars': 0})
accumulator.Reload()
scalar_tags = accumulator.Tags().get('scalars', [])
if not scalar_tags:
    raise RuntimeError('No scalar metrics were found in this event file.')
print('Scalar tags:')
for tag in scalar_tags:
    print(' -', tag)


In [ ]:
import csv, json, zipfile
from datetime import datetime, timezone

rows = []
latest_by_tag = {}
for tag in scalar_tags:
    events = accumulator.Scalars(tag)
    for event in events:
        rows.append({'tag': tag, 'step': int(event.step), 'value': float(event.value)})
    if events:
        last = max(events, key=lambda item: item.step)
        latest_by_tag[tag] = {'step': int(last.step), 'value': float(last.value)}

all_metrics_csv = Path('cifar10_all_epoch_metrics.csv')
with all_metrics_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['tag', 'step', 'value'])
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda item: (item['step'], item['tag'])))

def select_tag(fragment):
    matches = [tag for tag in scalar_tags if fragment in tag.lower()]
    return next((tag for tag in matches if tag.lower().split('/')[-1] == fragment), matches[0] if matches else None)

acc_tag = select_tag('val_acc')
loss_tag = select_tag('val_loss')
if acc_tag is None:
    raise RuntimeError(f'No validation/test accuracy tag found. Available tags: {scalar_tags}')
final_acc = latest_by_tag[acc_tag]
summary = {
    'run': 'cifar10_exp01_pepita_1hidden',
    'event_file': event_path.name,
    'metric_interpretation': 'The project validation hook uses the CIFAR-10 test loader.',
    'last_logged_epoch': final_acc['step'] + 1,
    'accuracy_tag': acc_tag,
    'accuracy_fraction': final_acc['value'],
    'accuracy_percent': 100.0 * final_acc['value'],
    'loss_tag': loss_tag,
    'loss': latest_by_tag[loss_tag]['value'] if loss_tag else None,
    'all_final_scalars': latest_by_tag,
    'extracted_utc': datetime.now(timezone.utc).isoformat(),
}
summary_json = Path('cifar10_final_metrics.json')
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f"Last logged epoch: {summary['last_logged_epoch']}")
print(f"Final logged test accuracy: {summary['accuracy_percent']:.2f}% ({summary['accuracy_fraction']:.6f})")
if summary['loss'] is not None:
    print(f"Final logged test loss: {summary['loss']:.6f}")

archive = Path('cifar10_metrics_for_report.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(all_metrics_csv)
    zf.write(summary_json)
print('Created:', archive)

In [ ]:
files.download(str(archive))